In [ ]:
# Colab reproducibility — uncomment to clone the repo and enter this folder:
# !git clone https://github.com/JuliettKhar/mlops-zoomcamp-exercises.git
# %cd mlops-zoomcamp-exercises/05-ml-monitoring

In [ ]:
# Colab setup — uncomment to install packages not preinstalled in Colab:
# !pip install evidently==0.7.21

In [1]:
import datetime
import pandas as pd

from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount

from joblib import load

In [ ]:
ref_data = pd.read_parquet("data/reference.parquet")
current_data = pd.read_parquet("data/green_tripdata_2026-02.parquet")

with open("models/lin_reg.bin", "rb") as f_in:
    model = load(f_in)

In [3]:
target = "duration_min"

num_features = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
]

cat_features = [
    "PULocationID",
    "DOLocationID"
]

features = num_features + cat_features
monitoring_columns = features + ["prediction"]

In [4]:
current_data["duration_min"] = (
    current_data.lpep_dropoff_datetime - current_data.lpep_pickup_datetime
).dt.total_seconds() / 60

current_data = current_data[
    (current_data.duration_min >= 0) &
    (current_data.duration_min <= 60)
]

current_data = current_data[
    (current_data.passenger_count > 0) &
    (current_data.passenger_count <= 8)
]

In [ ]:
# Выбираем проблемный день:
problematic_data = current_data.loc[
    (current_data.lpep_pickup_datetime >= datetime.datetime(2026, 2, 2, 0, 0)) &
    (current_data.lpep_pickup_datetime < datetime.datetime(2026, 2, 3, 0, 0))
].copy()

In [ ]:
# Добавляем prediction:

ref_data = ref_data.copy()
problematic_data = problematic_data.copy()

ref_data["prediction"] = model.predict(ref_data[features].fillna(0))
problematic_data["prediction"] = model.predict(problematic_data[features].fillna(0))

In [ ]:
# ColumnMapping
data_definition = DataDefinition(
    numerical_columns=num_features + ["prediction"],
    categorical_columns=cat_features
)

In [ ]:
# Создаём Dataset:
reference_dataset = Dataset.from_pandas(
    ref_data[monitoring_columns],
    data_definition=data_definition
)

current_dataset = Dataset.from_pandas(
    problematic_data[monitoring_columns],
    data_definition=data_definition
)

In [ ]:
# Report
report = Report([
    ValueDrift(column="prediction"),
    DataDriftPreset()
])

snapshot = report.run(
    reference_data=reference_dataset,
    current_data=current_dataset
)

In [18]:
result = snapshot.dict()

In [ ]:
# TestSuite
prediction_drift_score = next(
    m["value"]
    for m in result["metrics"]
    if m["config"].get("column") == "prediction"
)

num_drifted_columns = next(
    m["value"]["count"]
    for m in result["metrics"]
    if m["metric_name"].startswith("DriftedColumnsCount")
)

share_drifted_columns = next(
    m["value"]["share"]
    for m in result["metrics"]
    if m["metric_name"].startswith("DriftedColumnsCount")
)

In [13]:
print("Prediction drift score:", prediction_drift_score)
print("Number of drifted columns:", num_drifted_columns)
print("Share of drifted columns:", share_drifted_columns)

Prediction drift score: 0.05221581015663839
Number of drifted columns: 2.0
Share of drifted columns: 0.2857142857142857


In [ ]:
prediction_drift_threshold = 0.1
dataset_drift_share_threshold = 0.5

prediction_drift_test = prediction_drift_score <= prediction_drift_threshold
dataset_drift_test = share_drifted_columns <= dataset_drift_share_threshold

print("Prediction drift test:", "PASSED" if prediction_drift_test else "FAILED")
print("Dataset drift test:", "PASSED" if dataset_drift_test else "FAILED")

Prediction drift test: PASSED
Dataset drift test: PASSED


In [15]:
test_results = pd.DataFrame([
    {
        "test": "Prediction drift",
        "value": prediction_drift_score,
        "threshold": prediction_drift_threshold,
        "status": "PASSED" if prediction_drift_test else "FAILED"
    },
    {
        "test": "Dataset drift share",
        "value": share_drifted_columns,
        "threshold": dataset_drift_share_threshold,
        "status": "PASSED" if dataset_drift_test else "FAILED"
    }
])

test_results

,test,value,threshold,status
0,Prediction drift,0.052216,0.1,PASSED
1,Dataset drift share,0.285714,0.5,PASSED
